# AI 算力主题内动量因子检验

## 研究问题

检验 `ai_compute` 主题内部的短中期价格动量是否具有横截面收益解释力。

核心问题：在 AI 算力主题股票池中，过去 20 个交易日表现更强的股票，未来 5 个交易日是否继续获得更高收益？

本研究定位为单因子 baseline，不处理组合优化、交易成本、仓位约束和实盘执行问题。

## 研究链路

主题股票池 -> 复权价格矩阵 -> 历史收益 -> 动量因子 -> 未来收益 -> IC 检验 -> 分组收益 -> 结论

本 notebook 关注因子有效性检验，而不是策略净值回测。判断重点放在 IC 方向、IC 稳定性、分组收益单调性和样本稳健性。

## 参数设定

- 主题：`ai_compute`
- 价格字段：前复权收盘价
- 因子定义：过去 20 个交易日累计收益
- 预测收益：未来 5 个交易日收益
- 横截面检验：Spearman IC
- 分组方式：视样本数量使用 3 组或 5 组

初始版本固定参数，先建立可复现的 baseline。后续扩展到 60 日动量、未来 20 日收益，并按 subtrack 拆分研究景气传导。

In [2]:
import pandas as pd
import numpy as np

from utils.data import load_daily
from utils.themes import get_codes, get_stocks

THEME_ID = 'ai_compute'

START_DATE = '2021-01-01'
END_DATE = '2026-05-15'

ADJ = 'qfq'
PRICE_COL = 'close'

FACTOR_NAME = 'mom_20d'
LOOKBACK_WINDOW = 20
FORWARD_WINDOW = 5

IC_METHOD = 'spearman'
N_GROUPS = 5
MIN_STOCKS_PER_DATE = 5


## 1. 样本股票池

样本来自 `config/themes.yaml` 中的 `ai_compute` 主题。

记录项：

- 股票数量
- 股票代码与名称
- 是否存在明显非核心标的
- 是否存在上市时间过短、停牌较多、数据缺失较多的样本

样本容量会直接影响横截面 IC 和分组收益的可靠性。若有效样本过少，优先使用 IC 检验，分组收益仅作辅助观察。

In [14]:
stocks = get_stocks(THEME_ID)
codes = get_codes(THEME_ID)

stock_df = pd.DataFrame(stocks)

print(f'股票数量: {len(codes)}')
display(stock_df.head())

股票数量: 84


,code,name,subtrack
0,688256.SH,寒武纪,chip
1,688041.SH,海光信息,chip
2,300474.SZ,景嘉微,chip
3,688047.SH,龙芯中科,chip
4,688521.SH,芯原股份,chip


## 2. 价格矩阵

将个股前复权收盘价整理为交易日 × 股票代码的宽表。

目标结构：

```text
trade_date   688256.SH   688041.SH   ...
2023-01-03      100.12       35.21
2023-01-04      101.30       34.90
```

数据检查：

- 日期索引是否升序且唯一
- 股票列是否与样本池一致
- 缺失值分布是否集中在上市前或停牌期间
- 单只股票有效交易日数量是否明显偏少

本阶段不做复杂填充，优先保留原始缺失结构，避免引入额外假设。

## 3. 日收益率

由价格矩阵计算日收益率，用于基础数据诊断和后续异常检查。

基础定义：

```python
ret_1d = prices.pct_change()
```

诊断项：

- 单日极端收益分布
- 长时间无收益或价格不变的股票
- 缺失收益是否与价格缺失一致

第一版不做去极值和标准化。若后续发现极端值主导结果，再单独加入稳健性处理。

## 4. 动量因子

因子定义为过去 20 个交易日累计收益：

```python
mom_20d = prices.pct_change(20)
```

经济含义：过去一个月表现越强，因子值越高；过去一个月表现越弱，因子值越低。

检查项：

- 因子矩阵形状是否与价格矩阵一致
- 每个交易日有效因子样本数
- 因子横截面分布是否被少数极端值主导
- 前 20 个交易日无有效因子属于正常现象

## 5. 未来收益

预测目标定义为未来 5 个交易日收益：

```python
fwd_ret_5d = prices.shift(-5) / prices - 1
```

该写法将未来收益对齐到当前交易日。即 `t` 日的因子值，对应 `t` 日之后 5 个交易日的收益。

关键约束：

- 因子只能使用 `t` 日及以前可观察数据
- 未来收益只作为检验标签，不参与因子构造
- 最后 5 个交易日没有未来收益，属于正常缺失

## 6. 因子检验样本表

将因子矩阵和未来收益矩阵整理为长表，用于横截面统计。

目标结构：

```text
trade_date   ts_code     factor   forward_ret
2023-01-03   688256.SH   0.123    0.045
2023-01-03   688041.SH  -0.031    0.012
```

质量检查：

- `factor` 与 `forward_ret` 是否按同一 `trade_date / ts_code` 对齐
- 去除缺失后样本量
- 每个交易日有效横截面数量
- 是否存在单日样本数过少导致 IC 不稳定

## 7. IC 检验

每日在横截面上计算因子值与未来收益的 Spearman 相关系数。

统计指标：

- IC 均值
- IC 标准差
- ICIR：IC 均值 / IC 标准差
- IC 胜率：IC > 0 的交易日比例
- IC 时间序列与滚动均值

解释标准：

- IC 均值为正：动量排序与未来收益整体同向
- IC 胜率高：方向稳定性较好
- IC 波动大：结果可能依赖行情阶段或样本结构

单个主题内样本数有限，IC 绝对值不宜机械套用全市场经验阈值。

## 8. 分组收益

按每日因子值排序分组，观察高因子组是否获得更高未来收益。

分组规则：

- 样本充足时使用 5 组：Q1 最低，Q5 最高
- 样本较少时使用 3 组或 top/bottom 分组

观察指标：

- 各组平均未来收益
- Q5 - Q1 多空差值
- 分组收益是否单调
- 分组结果是否被少数日期或少数股票主导

分组收益用于解释因子形态，不等同于可交易策略收益。

## 9. 研究结论

结论需要回答以下问题：

1. `ai_compute` 主题内部 20 日动量对未来 5 日收益是否具有正向解释力？
2. IC 表现是否稳定，还是主要集中在少数阶段？
3. 分组收益是否呈现从低因子组到高因子组的单调关系？
4. 结果是否受样本数量、上市时间、停牌、极端个股或行情阶段影响？
5. 下一轮稳健性检验应扩展哪些维度？

结论模板：

```text
结论：
- 在 ai_compute 样本内，20 日动量对未来 5 日收益【有 / 无 / 不稳定】正向解释力。
- IC 均值为 ___，ICIR 为 ___，IC 胜率为 ___。
- 分组收益表现为 ___，Q5-Q1 为 ___。
- 当前结论的主要限制是 ___。
- 下一步验证：60 日动量、未来 20 日收益、按 subtrack 拆分景气传导。
```